# In this notebook we are creating the bootstrap dataset from our dataset in order to  show the robustness of our methods 

In [11]:
import pandas as pd
from tqdm import tqdm

Configurations:

In [ ]:
START_DATE=pd.to_datetime('2024-07-23')
SUBJECTS = ['math','stackoverflow','ubuntu','english','latex']
LLMS = [
"EleutherAI-pythia-6.9b",
"meta-llama-Llama-3.1-8B",
"meta-llama-Meta-Llama-3-8B-Instruct"
]
NUMBER_OF_SAMPLES = 50
SIZE_OF_SUBSET = 1000
PATH_TO_DATA=f"""../results/stackexchange_{{sub}}_combined/{{llm}}/aligned.parquet"""

In [ ]:

for llm in LLMS:
    all_genAI_data=[]
    for sub in SUBJECTS:
        print(f'Loading data for subject: {sub}')
        
        # Load genAI data
        genAI_data_sub = pd.read_parquet(PATH_TO_DATA.format(sub=sub, llm=llm))
        if 'CreationDate' not in genAI_data_sub.columns or 'ViewCount' not in genAI_data_sub.columns:
            genAI_data_sub = genAI_data_sub.rename(columns={'Question_Creation_Date': 'CreationDate', 'QuestionViewCount': 'ViewCount'})
        genAI_data_sub["CreationDate"] = pd.to_datetime(genAI_data_sub["CreationDate"])
        genAI_data_sub=genAI_data_sub[genAI_data_sub['CreationDate']>=START_DATE]
        genAI_data_sub['subject'] = sub  # Add subject identifier
        all_genAI_data.append(genAI_data_sub)
    # Concatenate all dataframes
    sub='united'
    genAI_data = pd.concat(all_genAI_data, ignore_index=True)
    genAI_data['year_week'] = genAI_data['CreationDate'].dt.to_period('W')
    weeks = sorted(genAI_data['year_week'].unique())
    for week_index, week in tqdm(enumerate(weeks), desc="Processing weeks"):
        week_start = week.start_time
        week_end = week.end_time
        # Filter genAI data for the current week
        genAI_weekly = genAI_data[(genAI_data['CreationDate'] >= week_start) & (genAI_data['CreationDate'] < week_end)]
        for i in range(NUMBER_OF_SAMPLES):
            
            # Sample questions for the current week
            SIZE_OF_SUBSET=len(genAI_weekly)//2
            genAI_sampled = genAI_weekly.sample(n=SIZE_OF_SUBSET if len(genAI_weekly) > 0 else 0, random_state=i)
        
            # Save the sampled data to a new Parquet file
            genAI_sampled.to_parquet(f'bootstrap_dataset/bootstrap_dataset_{sub}_{llm}_sample_{i+1}_week_{week_index+1}.parquet', index=False)